# IEEE-CIS Fraud Detection: baseline

Сравним DummyClassifier, Random Forest и XGBoost на одном временном split. Основная метрика — ROC-AUC, дополнительная — PR-AUC.

In [41]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)

RANDOM_STATE = 42
TARGET = "isFraud"
ID_COLUMN = "TransactionID"
SECONDS_PER_DAY = 86_400

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent
data_dir = project_root / "data"

## 1. Загрузка данных

Используем transaction-, identity- и 108 выбранных `V`-признаков. Отбор `V` здесь повторно не запускается.

In [42]:
transaction_columns = [
    "TransactionID", "TransactionDT", "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "dist1", "dist2",
    "P_emaildomain", "R_emaildomain",
    *[f"C{i}" for i in range(1, 15)],
    *[f"D{i}" for i in range(1, 16)],
    *[f"M{i}" for i in range(1, 10)],
]

selected_v_columns = """
V281 V282 V283 V289 V296 V301 V314 V1 V3 V4 V6 V8 V11 V13 V14 V17 V20
V23 V26 V27 V30 V36 V37 V40 V41 V44 V47 V48 V54 V55 V56 V59 V62 V65 V67
V68 V70 V76 V78 V80 V82 V86 V88 V89 V91 V127 V98 V99 V107 V108 V109 V111
V114 V115 V117 V120 V121 V123 V129 V130 V131 V138 V139 V142 V147 V155 V162
V165 V160 V166 V203 V207 V173 V169 V171 V174 V175 V180 V185 V198 V210 V209
V264 V223 V240 V241 V260 V277 V220 V221 V234 V238 V250 V271 V307 V284 V285
V286 V291 V303 V305 V309 V310 V312 V320 V332 V325 V335
""".split()

model_transaction_columns = transaction_columns + selected_v_columns

categorical_columns = [
    "ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain",
    *[f"M{i}" for i in range(1, 10)],
    "id_12", "id_15", "id_16", "id_23", "id_27", "id_28",
    "id_29", "id_30", "id_31", "id_33", "id_34", "id_35",
    "id_36", "id_37", "id_38", "DeviceType", "DeviceInfo",
]
identity_columns = [
    *[f"id_{i:02d}" for i in range(1, 10)],
    *[f"id_{i}" for i in range(10, 39)],
]

dtypes = {column: "float32" for column in model_transaction_columns + identity_columns}
dtypes.update({column: "category" for column in categorical_columns})
dtypes[ID_COLUMN] = "int32"
dtypes[TARGET] = "int8"

In [43]:
transactions = pd.read_csv(
    data_dir / "train_transaction.csv",
    index_col=ID_COLUMN,
    usecols=model_transaction_columns + [TARGET],
    dtype=dtypes,
)
identity = pd.read_csv(
    data_dir / "train_identity.csv",
    index_col=ID_COLUMN,
    dtype=dtypes,
)

X = transactions.join(identity, how="left")
y = X.pop(TARGET)
del transactions, identity


print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Fraud rate: {y.mean():.2%}")
print(f"X memory: {memory_mb:.1f} MB")

X shape: (590540, 201)
y shape: (590540,)
Fraud rate: 3.50%
X memory: 404.6 MB


## 2. Временной split

Случайный split смешал бы прошлое и будущее, поэтому делим данные по дням:

```text
train: 1–92 | gap: 93–122 | validation: 123–152 | holdout: 153–182
```

Final holdout пока не используем.

In [44]:
relative_day = (X["TransactionDT"] // SECONDS_PER_DAY).astype("int16")

time_masks = {
    "dev_train": relative_day.between(1, 92),
    "dev_gap": relative_day.between(93, 122),
    "dev_validation": relative_day.between(123, 152),
    "final_holdout": relative_day.between(153, 182),
}

In [45]:
split_summary = pd.DataFrame(
    {
        "rows": int(mask.sum()),
        "first_day": int(relative_day.loc[mask].min()),
        "last_day": int(relative_day.loc[mask].max()),
        "fraud_rate": y.loc[mask].mean(),
    }
    for mask in time_masks.values()
)
split_summary.index = time_masks.keys()
split_summary["fraud_rate"] = split_summary["fraud_rate"].map("{:.2%}".format)
display(split_summary)

,rows,first_day,last_day,fraud_rate
dev_train,326186,1,92,3.38%
dev_gap,93880,93,122,4.00%
dev_validation,85044,123,152,3.39%
final_holdout,85430,153,182,3.50%


In [46]:
dev_train_mask = time_masks["dev_train"]
dev_valid_mask = time_masks["dev_validation"]
X_dev_train = X.loc[dev_train_mask]
y_dev_train = y.loc[dev_train_mask]
X_dev_valid = X.loc[dev_valid_mask]
y_dev_valid = y.loc[dev_valid_mask]

print("Development train:", X_dev_train.shape)
print("Development validation:", X_dev_valid.shape)

Development train: (326186, 201)
Development validation: (85044, 201)


## 3. Кодирование категорий

`OrdinalEncoder` обучается только на train. Новые категории получают код `-1`, пропуски — `-2`.

In [47]:
cat_columns = [column for column in categorical_columns if column in X_dev_train.columns]
numeric_columns = [column for column in X_dev_train.columns if column not in cat_columns]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
                encoded_missing_value=-2,
                dtype=np.float32,
            ),
            cat_columns,
        ),
        ("numeric", "passthrough", numeric_columns),
    ],
    verbose_feature_names_out=False,
)

## 4. Метрики

ROC-AUC показывает качество ранжирования. PR-AUC дополнительно учитываем из-за редкого положительного класса.

In [48]:
def calculate_metrics(model_name, y_train, train_proba, y_valid, valid_proba, fit_seconds):
    train_roc_auc = roc_auc_score(y_train, train_proba)
    valid_roc_auc = roc_auc_score(y_valid, valid_proba)
    return {
        "model": model_name,
        "train_roc_auc": train_roc_auc,
        "valid_roc_auc": valid_roc_auc,
        "roc_auc_gap": train_roc_auc - valid_roc_auc,
        "train_pr_auc": average_precision_score(y_train, train_proba),
        "valid_pr_auc": average_precision_score(y_valid, valid_proba),
        "fit_seconds": fit_seconds,
    }

experiment_results = []

## 5. DummyClassifier

Простая проверка: ROC-AUC случайного baseline должен быть около 0.5.

In [49]:
dummy_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", DummyClassifier(strategy="prior")),
])

fit_started = perf_counter()
dummy_pipeline.fit(X_dev_train, y_dev_train)
dummy_fit_seconds = perf_counter() - fit_started

dummy_train_proba = dummy_pipeline.predict_proba(X_dev_train)[:, 1]
dummy_valid_proba = dummy_pipeline.predict_proba(X_dev_valid)[:, 1]

experiment_results.append(calculate_metrics(
    "DummyClassifier", y_dev_train, dummy_train_proba,
    y_dev_valid, dummy_valid_proba, dummy_fit_seconds,
))

## 6. Random Forest

Первый нелинейный baseline без подбора параметров.

In [50]:
rf = RandomForestClassifier(
    n_estimators=300, 
    max_depth=15,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", rf),
])

fit_started = perf_counter()
rf_pipeline.fit(X_dev_train, y_dev_train)
rf_fit_seconds = perf_counter() - fit_started

rf_train_proba = rf_pipeline.predict_proba(X_dev_train)[:, 1]
rf_valid_proba = rf_pipeline.predict_proba(X_dev_valid)[:, 1]

experiment_results.append(calculate_metrics(
    "Random Forest", y_dev_train, rf_train_proba,
    y_dev_valid, rf_valid_proba, rf_fit_seconds,
))

## 7. XGBoost

Используем тот же encoder. Validation нужен для early stopping.

In [51]:
xgb_preprocessor = clone(preprocessor)
xgb_clf = XGBClassifier(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.01,
    eval_metric="auc", 
    tree_method="hist", 
    early_stopping_rounds=50,
    verbosity=0, 
    n_jobs=-1, 
    random_state=RANDOM_STATE
)

fit_started = perf_counter()
X_dev_train_xgb = xgb_preprocessor.fit_transform(X_dev_train)
X_dev_valid_xgb = xgb_preprocessor.transform(X_dev_valid)

xgb_clf.fit(
    X_dev_train_xgb, 
    y_dev_train,
    eval_set=[(X_dev_valid_xgb, y_dev_valid)], 
    verbose=False
)

xgb_fit_seconds = perf_counter() - fit_started

xgb_train_proba = xgb_clf.predict_proba(X_dev_train_xgb)[:, 1]
xgb_valid_proba = xgb_clf.predict_proba(X_dev_valid_xgb)[:, 1]

experiment_results.append(calculate_metrics(
    "XGBoost", y_dev_train, xgb_train_proba,
    y_dev_valid, xgb_valid_proba, xgb_fit_seconds,
))

print(f"Best iteration: {xgb_clf.best_iteration}")

Best iteration: 1999


## 8. Сравнение baseline-моделей

In [52]:
experiment_table = (
    pd.DataFrame(experiment_results)
    .sort_values("valid_roc_auc", ascending=False)
    .reset_index(drop=True)
)

display(experiment_table.round(4))

,model,train_roc_auc,valid_roc_auc,roc_auc_gap,train_pr_auc,valid_pr_auc,fit_seconds
0,XGBoost,0.9297,0.8815,0.0482,0.6826,0.4813,140.2203
1,Random Forest,0.9214,0.8704,0.0510,0.6594,0.4415,48.7434
2,DummyClassifier,0.5000,0.5000,0.0000,0.0338,0.0339,1.2091


### Вывод

Random Forest получил ROC-AUC 0.8704 и PR-AUC 0.4415. XGBoost — 0.8815 и 0.4813. Дальше используем XGBoost.

## 9. Проверка на первом фолде

Повторим XGBoost на более раннем периоде: train 1–62, gap 63–92, validation 93–122.

In [53]:
fold1_masks = {
    "train": relative_day.between(1, 62),
    "gap": relative_day.between(63, 92),
    "validation": relative_day.between(93, 122),
}

fold1_summary = pd.DataFrame(
    {
        "rows": int(mask.sum()),
        "first_day": int(relative_day.loc[mask].min()),
        "last_day": int(relative_day.loc[mask].max()),
        "fraud_rate": y.loc[mask].mean(),
    }
    for mask in fold1_masks.values()
)

fold1_summary.index = fold1_masks.keys()
fold1_summary["fraud_rate"] = fold1_summary["fraud_rate"].map("{:.2%}".format)
display(fold1_summary)

X_fold1_train = X.loc[fold1_masks["train"]]
y_fold1_train = y.loc[fold1_masks["train"]]
X_fold1_valid = X.loc[fold1_masks["validation"]]
y_fold1_valid = y.loc[fold1_masks["validation"]]


print("Fold 1 train:", X_fold1_train.shape)
print("Fold 1 validation:", X_fold1_valid.shape)

,rows,first_day,last_day,fraud_rate
train,229906,1,62,3.16%
gap,96280,63,92,3.92%
validation,93880,93,122,4.00%


Fold 1 train: (229906, 201)
Fold 1 validation: (93880, 201)


In [54]:
fold1_preprocessor = clone(preprocessor)
xgb_fold1 = clone(xgb_clf)

fit_started = perf_counter()
X_fold1_train_xgb = fold1_preprocessor.fit_transform(X_fold1_train)
X_fold1_valid_xgb = fold1_preprocessor.transform(X_fold1_valid)

xgb_fold1.fit(
    X_fold1_train_xgb, 
    y_fold1_train,
    eval_set=[(X_fold1_valid_xgb, y_fold1_valid)], 
    verbose=False
)
fold1_fit_seconds = perf_counter() - fit_started

fold1_train_proba = xgb_fold1.predict_proba(X_fold1_train_xgb)[:, 1]
fold1_valid_proba = xgb_fold1.predict_proba(X_fold1_valid_xgb)[:, 1]

fold1_result = calculate_metrics(
    "XGBoost", 
    y_fold1_train, 
    fold1_train_proba,
    y_fold1_valid, 
    fold1_valid_proba, 
    fold1_fit_seconds
)

fold1_result["experiment"] = 'baseline'
fold1_result["fold"] = "fold_1"
fold1_result["best_iteration"] = xgb_fold1.best_iteration

print(f"Fold 1 best iteration: {xgb_fold1.best_iteration}")

Fold 1 best iteration: 1270


In [55]:
fold2_result = experiment_results[-1].copy()

fold2_result["experiment"] = 'baseline'
fold2_result["fold"] = "fold_2"
fold2_result["best_iteration"] = xgb_clf.best_iteration

xgb_fold_table = pd.DataFrame([fold1_result, fold2_result])[
    [
        "experiment", "fold", "train_roc_auc", "valid_roc_auc", "roc_auc_gap",
        "train_pr_auc", "valid_pr_auc", "best_iteration", "fit_seconds",
    ]
]

display(xgb_fold_table.round(4))

,experiment,fold,train_roc_auc,valid_roc_auc,roc_auc_gap,train_pr_auc,valid_pr_auc,best_iteration,fit_seconds
0,baseline,fold_1,0.9246,0.8813,0.0434,0.6580,0.5229,1270,95.6443
1,baseline,fold_2,0.9297,0.8815,0.0482,0.6826,0.4813,1999,140.2203


### Вывод по фолдам

ROC-AUC почти одинаковый: 0.8813 и 0.8815. PR-AUC отличается сильнее из-за разной доли fraud. Новые признаки будем проверять на обоих фолдах.

## Следующий шаг

Эксперименты с признаками находятся в `03_feature_engineering.ipynb`.